# The workflow around Insta360 X3 video

Lots of stuff to accomplish

- Copy the BBlogs and Videos to their directories.
- Read the metadata, including the IMU data.
- Compute the poses for the videos.
- Recompress and/or combine the videos.
    - Save this in the web directory.
- Extract the boat brain logs which align with the videos.
    - Sync them accurately
- Copy the poses and logs to the web directory.
- Create the videos.json file in the web dir.
- Sync the web dir to cloud.



The new framework I am trying is really a simplification.  All the files have the same basename
(which comes from the X3).  

```
-rw-r--r--  1 viola  staff         2368 May 17 09:29 VID_20240417_183646_00_003.exif
-rw-r--r--  1 viola  staff         4729 May 17 10:14 VID_20240417_183646_00_003.ffprobe
-rw-r--r--  1 viola  staff     34930240 May 17 09:29 VID_20240417_183646_00_003.imu
-rwx------@ 1 viola  staff  10942392580 Apr 17 19:05 VID_20240417_183646_00_003.insv
-rw-r--r--  1 viola  staff      3124816 May 20 09:43 VID_20240417_183646_00_003.log
-rw-r--r--  1 viola  staff      1392880 May 20 08:57 VID_20240417_183646_00_003.poses
```

The first is the captured video (which is really an MP4 with extra metadata).

- The metadata output of exiftool on this file.
- The metadata output of ffprobe.
- The raw extracted IMU data.
    - Exiftool, by default, tries to print this out in a friendly format.  Its crazy slow (minutes).
      I hacked the exiftool to just dump the bytes directly (second?).
    - This data can be read directly in python (numpy), once again in seconds!
- The X3 video file.
- The synced sensor logs collected by Boat Brain
- The camera poses estimated from the IMU data.


## Insta360 accel gyro data

Videos are captured in 29 min blocks.  

      Duration: 00:29:00.94, start: 0.000000, bitrate: 40357 kb/s

There appears to be a 15 second gap between files?!?!

    VID_20240422_180317_10_001.insv
    VID_20240422_183232_00_002.insv

(This is supposedly well known.)

There is also a "create time" stored in the metadata (exiftool or ffprobe).  Interestingly, **it is
not the same at the filename**?!?!

The best documentation I have on the accel/gyro data is from exiftool.  The data is stored in a binary blob with

- timecode:  unsigned 64bit in microseconds
- accelerometer:  3 unsigned shorts
    - Recenter and then downscale by 1000
- gyro: 3 ushort, same as above

The data is available at 1000Hz.  Extraction with exiftool took forever (using Perl to print
floats).  So I hacked exiftool to write the binary blob directly to a file (very fast).  Conversion
using numpy is insanely fast!

## Timing

**Timecode sucks.**

I am struggling with timecode.  The imu timecode seems to continue rolling as you continue to capture
video.  And it is not clear how it relates to framenumber.

**Filenames and creation times are not the same. **

The current plan (implemented below), do a rough extract of the BB logs (which is not synced).  Then
use the compass (from BB) and the gyro (from X3) to compute an alignment.  

In general it appears to be about 15 seconds.

### GPS data

Its possible to sync your phone to the X3, and it will pull across GPS data.  This is potentially
nice because it includes GPS time as well.

There is also a GPS enabled remote for $75 bucks.





In [ ]:
import os
import numpy as np
import re

from Libraries.log_helper import LOG
from Libraries.global_variables import G
from Libraries.nbutils import display_markdown, display
import Libraries.race_logs as race_logs
import Libraries.metadata as m
import Libraries.chart as ch
from Libraries.quick_plot import plot as quick_plot
from Libraries.quick_plot import multi_plot, qpa


# Initialize for Seattle.
G.init_seattle()
LOG.set_logging_level('DEBUG')

import Libraries.virb_video_pose as virb_video_pose
import Libraries.insta360 as insta360
import Libraries.geometry as geo




In [ ]:
import importlib
importlib.reload(geo)
importlib.reload(insta360)


In [ ]:
big_df = insta360.read_parquet_files()


In [ ]:

recompute = False
insta360.compute_all_pose_files(recompute=recompute)


In [ ]:
files = sorted([f for f in os.listdir(insta360.VIDEO_DIR) if re.match("^VID.*_00_.*insv$", f)])[-10:]
files = [os.path.splitext(f)[0] for f in files]

files


In [ ]:
file = files[-2]
print(file)
_, _, nhdg, ngg = insta360.extract_bb_logs_for_video(file, big_df, offset=200, recompute=True)

In [ ]:
# ngg is shorter than nhdg, pad out so they are the same length
padded_ngg = np.zeros((nhdg.shape[0]))
offset = 101*10
padded_ngg[offset:offset+ngg.shape[0]] = ngg

In [ ]:
multi_plot(
    qpa(None, (nhdg, padded_ngg), "hdg ngg".split())
)

In [ ]:

insta360.extract_all_bb_logs(big_df, recompute=recompute, offset=200)



In [ ]:
importlib.reload(insta360)
insta360.copy_to_web_dir()
_ = insta360.create_videos_json()

In [ ]:
dfs = insta360.read_parquet_files()

In [ ]:
importlib.reload(insta360)

video_basename = 'VID_20240415_191624_00_004'
video_basename = 'VID_20240508_174040_00_013'
video_basename = 'VID_20240601_133719_00_011'
video_basename = 'VID_20240417_173815_00_001'

insta360.compute_video_poses(video_basename, recompute=True)
_, _ = insta360.extract_bb_logs_for_video(video_basename, dfs, recompute=True)

In [ ]:
import Libraries.race_logs as race_logs
file = '2024-04-17T17_17.parquet'
race_logs.read_log_file(os.path.join(insta360.PANDAS_DIRECTORY, file))

In [ ]:
vdf, lf = insta360.extract_bb_logs_for_video(video_basename, dfs, recompute=True)

In [ ]:
dfs

In [ ]:
import telemetry_parser

In [ ]:
VIDEO_DIR = "/Users/viola/Data/Insta360/VID"

# video_basename = 'VID_20240527_072545_00_002'
video_basename = "VID_20240508_193742_00_017"
video_basename = 'VID_20240415_191624_00_004'
VIDEO_FILE = os.path.join(VIDEO_DIR, video_basename + '.insv')
IMU_FILE = os.path.join(VIDEO_DIR, video_basename + '.imu')


In [ ]:

def extract_telemetry(video_file):
    tp = telemetry_parser.Parser(video_file)
    telemetry = tp.telemetry()
    LOG.logger.info(f'Model {tp.model}')
    return telemetry[0]['Metadata']


tp = telemetry_parser.Parser(VIDEO_FILE)

print('Model: ', tp.model)
ttt = tp.telemetry()

mmm = ttt[0]['Default']['Metadata']

In [ ]:


ttt, aaa, ggg = insta360.read_binary_imu_data(IMU_FILE)
# ttt, aaa, ggg = insta360.read_insta_imu_data(video_basename, shift=0)


In [ ]:
ggg

In [ ]:
mmm['first_frame_timestamp']/1000.0

In [ ]:
iii = np.searchsorted(ttt, mmm['first_frame_timestamp']/1000.0, side='right')
iii

In [ ]:
ttt[iii] - ttt[0]

In [ ]:
importlib.reload(insta360)
q_initial = geo.q_identity()

ngg = ggg * [1, 0, 0]

# q_poses = geo.estimate_pose(q_initial, aaa, ggg, accel_gain=0.0005, gyro_gain=0.0)
# q_poses = geo.estimate_pose(q_initial, aaa, ngg, accel_gain=0.0000, gyro_gain=1.0)
q_poses = geo.estimate_pose(q_initial, aaa, ggg, accel_gain=0.0005, gyro_gain=0.0)
insta360.write_pose_file(os.path.join(insta360.VIDEO_DIR, 'video.poses'), q_poses)


In [ ]:
# Extract the bblog data for each video.
importlib.reload(insta360)

big_df = insta360.read_parquet_files()


In [ ]:
import Libraries.race_logs as race_logs

file = '2024-04-15T17_17.parquet'

sdf = race_logs.read_log_file(os.path.join(insta360.PANDAS_DIRECTORY, file))

In [ ]:
# time string in mins:seconds  i.e. 1:23.456 convert to seconds
def time_to_seconds(time_str):
    mins, secs = time_str.split(':')
    return int(mins) * 60 + float(secs)

bb, ee = time_to_seconds('2:28.66'), time_to_seconds('2:38.385')
bb, ee, (ee + bb)/2

In [ ]:
importlib.reload(insta360)
importlib.reload(virb_video_pose)
video_basename = 'VID_20240415_191624_00_004'

shift = 2.2
# insta360.compute_video_poses(video_basename, recompute=True, shift=shift)
vdf, lf = insta360.extract_bb_logs_for_video(video_basename, big_df, shift=shift, recompute=True)

In [ ]:
df = vdf
msec_per_sample = 100
multiplier = 32767

log_file = "/tmp/insta360.log"

# with open(log_file, 'wb') as fs:
if True:
    rows, cols = df.shape
    header = np.array([rows, cols, msec_per_sample, multiplier], dtype=np.int32)
    values = df.values
    # get the maximum absolute value for each column
    max_values = np.max(values, axis=0).astype(np.float64)
    min_values = np.min(values, axis=0).astype(np.float64)
    print(max_values)
    print(min_values)
    values = (values - min_values) / (max_values - min_values)
    # scale to in16 range
    values = (values * multiplier).astype(np.uint16)


    # fs.write(header.tobytes())
    # fs.write(min_values.tobytes())
    # write data column by column (F is for fortran!)
    # fs.write(values.tobytes('F'))



In [ ]:
bbb = values[::1000, -1]
print(len(bbb))
# print each byte as an integer
for b in bbb:
    print(b, end=' ')

In [ ]:
ttt = 1713233723392.0 + 10
aaa = np.array([ttt]).astype(np.float32)
print(ttt)
print(aaa)
print(aaa.dtype)
print(aaa[0])

In [ ]:
vdf.row_times

In [ ]:
importlib.reload(insta360)

In [ ]:
video_df, video_start, video_end = insta360.extract_video_logs(video_basename, big_df, offset=30)
time_stamps, accel, gyro = insta360.read_insta_imu_data(video_basename, shift=2.2)

dhdg = np.diff(video_df.hdg, prepend=video_df.hdg.iloc[0])
dhdg[dhdg > 180] -= 360
dhdg[dhdg < -180] += 360

def normalize(signal):
    return (signal - np.mean(signal)) / np.std(signal)

# Downsample the gyro data to 10Hz (from 1000Hz).  Axis 1 is the vertical axis.
sgg = gyro[::100, 0]
xgg = np.linspace(0, sgg.shape[0]/10, len(sgg))

ngg = normalize(sgg)
nhdg = normalize(dhdg)

match = np.correlate(nhdg, 1*ngg, mode='valid')
delay = np.argmax(match)

In [ ]:
dhdg = np.diff(vdf.hdg, prepend=vdf.hdg.iloc[0])
dhdg[dhdg > 180] -= 360
dhdg[dhdg < -180] += 360



In [ ]:
multi_plot(
    qpa(None, (ngg,)),
    qpa(None, (nhdg,)),
    qpa(None, (dhdg,)),
)

In [ ]:
saa = 10*aaa[::100]
cvdf.loc[:saa.shape[0]-1, "hdg twd tws".split()] = saa

In [ ]:
sgg = 10*ggg[::100]
cvdf.loc[:sgg.shape[0]-1, "spd awa twa".split()] = sgg

In [ ]:
virb_video_pose.save_log_to_binary(cvdf, lf)

In [ ]:
# create a json file that lists all videos.





    


In [ ]:

# plot heading
multi_plot(
    qpa(cdf.row_seconds, (nhdg,),  'heading'),    
    qpa(xgg, (ngg, ),  'gyro1, gyro2, gyro3'),
)



In [ ]:
dhdg

In [ ]:
cgg = np.cumsum(gg1, axis=0) / 1000


multi_plot(
    qpa(times, (sgg[:, 0], sgg[:, 1], sgg[:, 2]), "g0, g1, g2"),
    qpa(times, (cgg[:, 0], cgg[:, 1], cgg[:, 2]), "cg0, cg1, cg2"),
)


## Observations

From the viewer:  

    // By convention, we follow the OpenGL framework for "right hand rule, x is horizontal (right),
    // y vertical (up), and looking down negative z".

Accelerometers are sort of backward.  When it is accelerating in the positive direction of the axis
they show negative numbers.  So the device sitting on the desk is showing negative acceleration in
the up direction.  

- a0 points up. Almost always negative 1G.  
    - [-1, 0, 0] is straight up.
    - This is -Y in opengl.
- a1 points forward.  Nose up is more negative. 
    - [0, -1, 0] is sitting on its back.
    - This is -Z in opengl (since we are looking down the negative Z direction)
- a2 points right/stbd (roll axis)  On port tack (rolled right) it is positive.
    - [0, 0, -1] is sitting on its left side
    - This is X in opengl.

Gyros are more arbitrary. The right hand rule says a positive twist is counter clockwise.

- g0 is yaw (around OpenGL Y), rotation around vertical.  Yawing right (to stbd) is positive.
    - Twist around -Y.
- g1 is roll (around OpenGL Z).  Rolling left (to port) is positive.
    - Twist around Z
- g2 is pitch (around OpenGL X).  Pitching down (nose down) is positive. 
    - Twist around -X


So if 

In [ ]:
def show_pose(q):
    pose_mat = geo.q_mat(q)
    i_hat = pose_mat[:, 0]
    j_hat = pose_mat[:, 1]
    k_hat = pose_mat[:, 2]
    LOG.logger.debug(f"Q  : {q}")
    LOG.logger.debug(f"rt : {i_hat}")
    LOG.logger.debug(f"up : {j_hat}")
    LOG.logger.debug(f"bw : {k_hat}")



## Observations

- a0 points up (yaw axis)  Almost always negative 1G.
- a1 points backward (pitch axis)  Nose down is more negative.
- a2 points right/stbd (roll axis)  On port tack it is negative

- g0 is roll.  Rolling right (to stbd) is positive.  Right hand rule say forward.
- g1 is yaw.  Turning right is negative.  RHR says up.
- g2 is pitch 